# **Motor de Posicionamento (Indoor Positioning System - IPS)**
<font size=3>
    
O objetivo desta fase é coletar os sinais de rádio Wi-Fi e os dados dos sensores do celular e convertê-los, em tempo real, nas coordenadas cartesianas $(X, Y)$ que indicam onde a pessoa está no prédio.

### **1. Captura do Sinal Wi-Fi no Android:**
<font size=3>
    
Para ler a intensidade do sinal Wi-Fi no Android nativo, a aplicação consulta a API do sistema operacional para obter a lista de roteadores visíveis.

* **Permissões Necessárias (no `AndroidManifest.xml`):**
    * `ACCESS_FINE_LOCATION` ou `ACCESS_COARSE_LOCATION`: O Android exige permissão de localização para varrer redes Wi-Fi por motivos de privacidade.
    * `CHANGE_WIFI_STATE` e `ACCESS_WIFI_STATE`: Para habilitar e consultar o estado do chip Wi-Fi.
    * `NEARBY_WIFI_DEVICES`: Necessário a partir do Android 13.
    <br>

* **Componente `WifiManager`:** É o serviço do sistema responsável por disparar a varredura e retornar os objetos `ScanResult`. Cada `ScanResult` traz:
    * `BSSID`: Endereço MAC do roteador.
    * `level`: O valor de **RSSI** em dBm (ex: $-65\text{ dBm}$). Quanto mais próximo de zero, mais forte é o sinal.
    <br>

* **Contorno do Limitation Throttle (*Foreground Service*):** Como o Android limita varreduras frequentes em *background*, a leitura de Wi-Fi deve rodar associada a um **Foreground Service** (um serviço nativo com uma notificação fixa na barra do celular).


### **2. A Matemática da Distância (Modelo *Log-Distance Path Loss*):**
<font size=3>
    
O valor de RSSI (dBm) não é uma distância direta. Para converter a força do sinal em uma distância $d$ em metros, utiliza-se a equação do **Modelo de Perda de Percurso (Log-Distance Path Loss)**:
    $$RSSI(d) = RSSI_0 - 10 \cdot n \cdot \log_{10}\left(\frac{d}{d_0}\right)$$

Isolando a distância $d$ (considerando $d_0 = 1\text{ metro}$):
    $$d = 10^{\left(\frac{RSSI_0 - RSSI}{10 \cdot n}\right)}$$

* $RSSI$: O sinal recebido no momento (ex: $-68\text{ dBm}$).
* $RSSI_0$: O sinal de referência medido a 1 metro de distância (ex: $-40\text{ dBm}$).
* $n$: O expoente de atenuação do ambiente (varie entre $2.0$ e $3.5$ conforme as paredes do prédio).

In [ ]:
import kotlin.math.pow

In [ ]:
fun calculateDistanceMeters(currentRssi: Int, referenceRssi1m: Int = -40, attenuationExponent: Double = 2.5): Double {
    val exponent = (referenceRssi1m - currentRssi) / (10.0 * attenuationExponent)
    return 10.0.pow(exponent)
}

### **3. Filtragem de Ruído (Suavização de RSSI):**
<font size=3>
    
O sinal Wi-Fi sofre interferências severas devido a reflexões nas paredes, móveis e presença de pessoas. Um valor de RSSI pode flutuar de $-55\text{ dBm}$ para $-70\text{ dBm}$ mesmo com o celular parado. Para evitar que a posição "pule" na tela, aplicamos um **Filtro de Média Móvel (Moving Average Filter)** sobre as últimas leituras de cada BSSID.

In [ ]:
class MovingAverageFilter(private val windowSize: Int = 5) {
    private val readingsHistory = mutableListOf<Int>()

    fun addAndFilter(newRssi: Int): Double {
        readingsHistory.add(newRssi)
        if (readingsHistory.size > windowSize) {
            readingsHistory.removeAt(0)
        }
        return readingsHistory.average()
    }
}

### **4. Algoritmo de Trilateração 2D:**
<font size=3>
    
Com a distância estimada até pelo menos 3 roteadores de posições $(x_i, y_i)$ conhecidas, montamos um sistema geométrico onde a posição do usuário $(x, y)$ é o ponto de interseção das três circunferências:

$$(x - x_1)^2 + (y - y_1)^2 = d_1^2$$

$$(x - x_2)^2 + (y - y_2)^2 = d_2^2$$

$$(x - x_3)^2 + (y - y_3)^2 = d_3^2$$

Subtraindo a primeira equação da segunda e da terceira, eliminamos os termos quadráticos ($x^2, y^2$) e obtemos um sistema linear da forma $A \cdot X = B$, resolvido pelo **Método dos Mínimos Quadrados**:

In [ ]:
data class Point2D(val x: Double, val y: Double)

fun trilateration2D(p1: Point2D, d1: Double, p2: Point2D, d2: Double, p3: Point2D, d3: Double): Point2D {
    val a = 2 * (p2.x - p1.x)
    val b = 2 * (p2.y - p1.y)
    val c = d1.toDouble().pow(2) - d2.toDouble().pow(2) - p1.x.pow(2) + p2.x.pow(2) - p1.y.pow(2) + p2.y.pow(2)

    val d = 2 * (p3.x - p1.x)
    val e = 2 * (p3.y - p1.y)
    val f = d1.toDouble().pow(2) - d3.toDouble().pow(2) - p1.x.pow(2) + p3.x.pow(2) - p1.y.pow(2) + p3.y.pow(2)

    val x = (c * e - f * b) / (a * e - d * b)
    val y = (a * f - d * c) / (a * e - d * b)

    return Point2D(x, y)
}

### **5. Odometria Pediátrica (PDR) e Fusão de Sensores:**
<font size=3>
    
Como as varreduras de Wi-Fi no Android têm um intervalo de alguns segundos, usamos os sensores de inércia do celular para atualizar o deslocamento do usuário em tempo real entre duas leituras de rádio.

* **Sensor de Detecção de Passos (`TYPE_STEP_DETECTOR`):** Dispara um evento do `SensorManager` toda vez que o usuário completa um passo.

* **Tamanho Médio do Passo ($L$):** Estimado em torno de $0.65\text{ metros}$ a $0.75\text{ metros}$ por passo.

* **Sensor de Orientação/Bússola (`TYPE_ROTATION_VECTOR`):** Fornece o azimute (ângulo $\theta$ em relação ao Norte magnético em radianos).

* **Atualização da Posição com PDR:**

$$x_{novo} = x_{anterior} + L \cdot \cos(\theta)$$

$$y_{novo} = y_{anterior} + L \cdot \sin(\theta)$$

### **Atividade prática:**
<font size=3>
    
Simular o Motor IPS:\
Escreva uma função que receba 3 sinais RSSI fictícios capturados no corredor, converta para distâncias, aplique o filtro e execute a trilateração para estimar a coordenada $(X, Y)$ do usuário.

<details>
<summary>Resposta:</summary>

<font size=2>
<br>
    
```kotlin
// 1. Definição da localização física conhecida de 3 Roteadores no prédio
val ap1 = Point2D(0.0, 0.0)   // AP 1 na origem
val ap2 = Point2D(10.0, 0.0)  // AP 2 a 10m no eixo X
val ap3 = Point2D(5.0, 10.0)  // AP 3 no topo do triângulo

// 2. Simulação de leituras de RSSI recebidas pelo celular
val rssiAp1 = -58 // Equivalente a aprox. 5.2 metros
val rssiAp2 = -58 // Equivalente a aprox. 5.2 metros
val rssiAp3 = -58 // Equivalente a aprox. 5.2 metros

// 3. Conversão de RSSI para distâncias em metros
val d1 = calculateDistanceMeters(rssiAp1, referenceRssi1m = -40, attenuationExponent = 2.5)
val d2 = calculateDistanceMeters(rssiAp2, referenceRssi1m = -40, attenuationExponent = 2.5)
val d3 = calculateDistanceMeters(rssiAp3, referenceRssi1m = -40, attenuationExponent = 2.5)

// 4. Estimativa da posição pelo algoritmo de Trilateração
val estimatedPosition = trilateration2D(ap1, d1, ap2, d2, ap3, d3)

println("--- RESULTADO DO MOTOR IPS ---")
println("Distâncias estimadas: d1=%.2fm, d2=%.2fm, d3=%.2fm".format(d1, d2, d3))
println("Posição estimada do usuário: X = %.2fm, Y = %.2fm".format(estimatedPosition.x, estimatedPosition.y))
```
<br>

</details>
